In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


# Get student enrollment
def get_enrollment_data(url, university_name):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except Exception as e:
        print(f"Request failed: {university_name}: {e}")
        return None
    
    soup = BeautifulSoup(response.text, 'html.parser')
    items = soup.select('div.numbers-item')
 
    for item in items:
        number_el = item.select_one('p.numbers-number')
        desc_el = item.select_one('p.numbers-description')
        
        if not number_el or not desc_el:
            continue
            
        number = number_el.text.strip().replace(",", "")
        description = desc_el.get_text(" ", strip=True)
        
        if number.isdigit() and "Traditional" in description:
            return int(number)
    
    return None

# Get weather data
def get_frozen_days(lat, lon, start_date, end_date):
  URL = (
      f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start_date}&end_date={end_date}"
        f"&daily=temperature_2m_min,snowfall_sum&timezone=auto"
  )

  try:
    response = requests.get(URL, timeout=15)
    response.raise_for_status()
  except Exception as e:
    print(f"Weather API error for {lat}, {lon}: {e}")
    return []

  weather_data = response.json()
  frozen_days = []


  dates = weather_data.get('daily', {}).get('time', [])
  temps = weather_data.get('daily', {}).get('temperature_2m_min', [])
  snowfall = weather_data.get('daily', {}).get('snowfall_sum', [])

  for d, t, s in zip(dates, temps, snowfall):
    if t is not None and s is not None:
       if t <= -7  or s >= 10:
          frozen_days.append(d)
          
  return frozen_days
 
  
universities = [
    {
    'name': 'WashU',
    'state': 'Missouri',
    'latitude': 38.648,
    'longitude': -90.305,
     'url': 'https://washu.edu/about-washu/university-facts/'},
]

start_date = '2026-01-01'
end_date = '2026-01-31'

results = []

for uni in universities:
  enrollment = get_enrollment_data(uni['url'], uni['name'])
  if enrollment is None:
    enrollment = 0

  frozen_days = get_frozen_days(
     uni['latitude'],
     uni['longitude'],
     start_date,
     end_date
  )

  
  results.append({
    'University Name': uni['name'],
    'State': uni['state'],
    'Number of Students Enrolled': enrollment,
    'Severe Weather Days (January 2026)': frozen_days,
    'Number of Severe Days': len(frozen_days),  
    'Student-Days Impacted': enrollment * len(frozen_days)
  })


df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
df